In [0]:
%pip install pandas folium osmnx networkx

In [0]:
# Import libraries
import googlemaps
import pandas as pd
from datetime import datetime
import folium
from folium import plugins

# Set up your Google Maps API key
# Get a free API key from: https://developers.google.com/maps/documentation/javascript/get-api-key
GOOGLE_MAPS_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your actual API key

# Initialize the Google Maps client
gmaps = googlemaps.Client(key=GOOGLE_MAPS_API_KEY)

In [0]:
%pip install geopy

In [0]:
import pandas as pd
from geopy.distance import geodesic

def calculate_distances(home_coords, stores):
    """
    Calculate straight-line distance from home to each ice cream store
    """
    results = []
    
    for store in stores:
        # Calculate geodesic distance using provided coordinates
        dist_miles = geodesic(
            (home_coords['lat'], home_coords['lng']),
            (store['lat'], store['lng'])
        ).miles
        dist_km = geodesic(
            (home_coords['lat'], home_coords['lng']),
            (store['lat'], store['lng'])
        ).km
        
        results.append({
            'Store Name': store['name'],
            'Address': store['address'],
            'Distance (miles)': round(dist_miles, 2),
            'Distance (km)': round(dist_km, 2),
            'Latitude': store['lat'],
            'Longitude': store['lng']
        })
    
    df = pd.DataFrame(results)
    df = df.sort_values('Distance (miles)')
    return df

# Calculate distances
results_df = calculate_distances(home_coords, ice_cream_stores)

# Display results
display(results_df)

In [0]:
# Your home coordinates
home_coords = {'lat': 42.2626, 'lng': -71.8023}
home_address = "Highland St, Worcester, MA"

# List of ice cream stores with coordinates
ice_cream_stores = [
    {
        "name": "Gibby's Famous Ice Cream",
        "address": "323 Greenwood St, Worcester, MA 01607",
        "lat": 42.2338,
        "lng": -71.7916
    },
    {
        "name": "Pinecroft Dairy and Restaurant",
        "address": "539 Prospect St, West Boylston, MA 01583",
        "lat": 42.3665,
        "lng": -71.8015
    },
    {
        "name": "Swirls and Scoops",
        "address": "100 Worcester St, North Grafton, MA 01536",
        "lat": 42.2286,
        "lng": -71.6857
    },
    {
        "name": "West End Creamery",
        "address": "481 Purgatory Rd, Whitinsville, MA 01588",
        "lat": 42.1118,
        "lng": -71.6662
    },
    {
        "name": "Rota Spring Farm",
        "address": "117 Chace Hill Rd, Sterling, MA 01564",
        "lat": 42.4359,
        "lng": -71.7601
    },
    {
        "name": "Murdock Farm and Dairy Bar",
        "address": "1143 Central St, Winchendon, MA 01475",
        "lat": 42.6876,
        "lng": -72.0437
    },
    {
        "name": "Shaw Farm Dairy",
        "address": "204 New Boston Rd, Dracut, MA 01826",
        "lat": 42.6762,
        "lng": -71.3020
    },
    {
        "name": "High Lawn Farm",
        "address": "535 Summer St, Lee, MA 01238",
        "lat": 42.3047,
        "lng": -73.2485
    },
    {
        "name": "Joyful Scoops",
        "address": "50 Harding St, Middleborough, MA 02346",
        "lat": 41.8945,
        "lng": -70.9080
    },
    {
        "name": "Alvin Rondeau's Dairy Bar",
        "address": "1335 Park St, Palmer, MA 01069",
        "lat": 42.1762,
        "lng": -72.3287
    }
]

In [0]:
import openrouteservice as ors

# 👉 GET YOUR FREE API KEY: https://openrouteservice.org/dev/#/signup
# Takes less than 1 minute - no credit card needed!
# Then paste your key below:

ORS_API_KEY = "YOUR_ORS_API_KEY_HERE"  # 👈 REPLACE THIS with your actual key

# Initialize OpenRouteService client
try:
    if ORS_API_KEY == "YOUR_ORS_API_KEY_HERE":
        raise Exception("Please add your API key above")
    
    ors_client = ors.Client(key=ORS_API_KEY)
    print("✅ SUCCESS! OpenRouteService connected!")
    print("   You'll get actual driving routes on your map.")
    print("   Re-run Cell 8 to see the real routes!")
except Exception as e:
    ors_client = None
    print("💡 TO GET ACTUAL DRIVING ROUTES (not straight lines):")
    print("   1. Visit: https://openrouteservice.org/dev/#/signup")
    print("   2. Sign up (FREE - takes 1 minute)")
    print("   3. Copy your API key from the dashboard")
    print("   4. Paste it above where it says 'YOUR_ORS_API_KEY_HERE'")
    print("   5. Re-run this cell (Shift+Enter)")
    print("")
    print("⚠️  Current status: Using straight lines (API key not set)")

In [0]:
def calculate_route_info(home, stores):
    """
    Calculate distance and duration from home to each ice cream store
    """
    results = []
    
    # Geocode home address
    home_geocode = gmaps.geocode(home)[0]
    home_location = home_geocode['geometry']['location']
    
    for store in stores:
        try:
            # Get directions
            directions = gmaps.directions(
                home,
                store['address'],
                mode="driving",
                departure_time=datetime.now()
            )
            
            if directions:
                # Extract relevant information
                leg = directions[0]['legs'][0]
                
                # Geocode store address
                store_geocode = gmaps.geocode(store['address'])[0]
                store_location = store_geocode['geometry']['location']
                
                results.append({
                    'Store Name': store['name'],
                    'Address': store['address'],
                    'Distance (miles)': round(leg['distance']['value'] / 1609.34, 2),
                    'Distance (km)': round(leg['distance']['value'] / 1000, 2),
                    'Duration (minutes)': round(leg['duration']['value'] / 60, 2),
                    'Duration (text)': leg['duration']['text'],
                    'Distance (text)': leg['distance']['text'],
                    'Latitude': store_location['lat'],
                    'Longitude': store_location['lng']
                })
        except Exception as e:
            print(f"Error processing {store['name']}: {str(e)}")
    
    # Create DataFrame and sort by duration
    df = pd.DataFrame(results)
    df = df.sort_values('Duration (minutes)')
    
    return df, home_location

# Calculate routes
results_df, home_coords = calculate_route_info(home_address, ice_cream_stores)

# Display results
display(results_df)

In [0]:
import folium
import requests
import time

# Create map with ACTUAL driving routes using OSRM (free public API)
def create_ice_cream_trail_map(home_coords, stores_df):
    """
    Create Ice Cream Trail map with actual driving routes using OSRM
    """
    # Vibrant CartoDB Voyager base map
    m = folium.Map(
        location=[42.2, -71.8],  # Center on Massachusetts
        zoom_start=8,  # Zoom out to show full state
        tiles='CartoDB Voyager'
    )
    
    # Add Massachusetts state outline using accurate GeoJSON data
    print("📍 Loading Massachusetts state boundary...")
    try:
        # Use US states GeoJSON (accurate boundary data)
        ma_geojson_url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
        
        # Custom style for Massachusetts
        def style_function(feature):
            if feature['properties']['name'] == 'Massachusetts':
                return {
                    'fillColor': '#e3f2fd',
                    'color': '#1976d2',
                    'weight': 3,
                    'fillOpacity': 0.15
                }
            else:
                # Hide other states
                return {
                    'fillColor': 'transparent',
                    'color': 'transparent',
                    'weight': 0,
                    'fillOpacity': 0
                }
        
        folium.GeoJson(
            ma_geojson_url,
            style_function=style_function,
            # Disable all interactions to prevent click rectangles
            highlight_function=None,
            tooltip=None,
            popup=None,
            show=True,
            overlay=True,
            control=False,
            smooth_factor=1.0
        ).add_to(m)
        
        print("  ✅ Massachusetts boundary loaded!")
    except Exception as e:
        print(f"  ⚠️ Could not load state boundary: {e}")
    
    # Color function
    def get_route_color(distance):
        if distance <= 5:
            return '#00cc00'  # Bright Green
        elif distance <= 15:
            return '#ff9900'  # Orange  
        elif distance <= 30:
            return '#ff6600'  # Dark Orange
        else:
            return '#cc0000'  # Red
    
    # Add home marker
    folium.Marker(
        [home_coords['lat'], home_coords['lng']],
        popup='<b style="font-size:16px">🏠 HOME BASE</b><br><i>Highland St, Worcester, MA</i>',
        icon=folium.Icon(color='red', icon='home', prefix='fa', icon_color='white'),
        tooltip='🏠 Start Here!'
    ).add_to(m)
    
    print("🚗 Fetching actual driving routes from OSRM...")
    
    # Draw routes with ACTUAL road paths
    route_num = 1
    successful_routes = 0
    
    for idx, row in stores_df.iterrows():
        distance = row['Distance (miles)']
        route_color = get_route_color(distance)
        
        try:
            # Get actual driving route from OSRM public API
            url = f"http://router.project-osrm.org/route/v1/driving/{home_coords['lng']},{home_coords['lat']};{row['Longitude']},{row['Latitude']}"
            params = {
                'overview': 'full',
                'geometries': 'geojson'
            }
            
            response = requests.get(url, params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                
                if data['code'] == 'Ok' and len(data['routes']) > 0:
                    # Extract the route geometry
                    route_coords = data['routes'][0]['geometry']['coordinates']
                    # Convert from [lon, lat] to [lat, lon] for folium
                    route_coords = [[coord[1], coord[0]] for coord in route_coords]
                    
                    # Get driving distance and duration
                    route_distance_meters = data['routes'][0]['distance']
                    route_duration_seconds = data['routes'][0]['duration']
                    route_miles = route_distance_meters / 1609.34
                    route_minutes = route_duration_seconds / 60
                    
                    # Draw the actual route
                    folium.PolyLine(
                        locations=route_coords,
                        color=route_color,
                        weight=4,
                        opacity=0.85,
                        popup=f"<b>{row['Store Name']}</b><br>🚗 {route_miles:.1f} miles<br>⏱️ {route_minutes:.0f} min drive",
                        tooltip=f"Route #{route_num}: {row['Store Name']} - {route_miles:.1f} mi"
                    ).add_to(m)
                    
                    successful_routes += 1
                    print(f"  ✅ Route {route_num}: {row['Store Name']} - {route_miles:.1f} miles")
                else:
                    # Fallback to straight line
                    print(f"  ⚠️ Route {route_num}: {row['Store Name']} - using straight line")
                    folium.PolyLine(
                        locations=[
                            [home_coords['lat'], home_coords['lng']],
                            [row['Latitude'], row['Longitude']]
                        ],
                        color=route_color,
                        weight=3,
                        opacity=0.6,
                        dash_array='5, 5',
                        popup=f"<b>{row['Store Name']}</b><br>📍 ~{distance} miles",
                        tooltip=f"Route #{route_num}: {row['Store Name']}"
                    ).add_to(m)
            else:
                # Fallback to straight line
                folium.PolyLine(
                    locations=[
                        [home_coords['lat'], home_coords['lng']],
                        [row['Latitude'], row['Longitude']]
                    ],
                    color=route_color,
                    weight=3,
                    opacity=0.6,
                    dash_array='5, 5',
                    popup=f"<b>{row['Store Name']}</b><br>📍 ~{distance} miles",
                    tooltip=f"Route #{route_num}: {row['Store Name']}"
                ).add_to(m)
            
            # Rate limiting - be nice to the free API
            time.sleep(0.3)
            
        except Exception as e:
            print(f"  ⚠️ Error for {row['Store Name']}: {str(e)}")
            # Fallback to straight line
            folium.PolyLine(
                locations=[
                    [home_coords['lat'], home_coords['lng']],
                    [row['Latitude'], row['Longitude']]
                ],
                color=route_color,
                weight=3,
                opacity=0.6,
                dash_array='5, 5',
                popup=f"<b>{row['Store Name']}</b><br>📍 ~{distance} miles",
                tooltip=f"Route #{route_num}: {row['Store Name']}"
            ).add_to(m)
        
        # Add store marker
        folium.Marker(
            [row['Latitude'], row['Longitude']],
            popup=f'''
            <div style="width:220px">
                <b style="font-size:16px">🍦 #{route_num}: {row['Store Name']}</b><br>
                <i>{row['Address']}</i><br>
                <hr style="margin:5px 0">
                📍 {distance} miles away
            </div>
            ''',
            icon=folium.Icon(color='blue', icon='ice-cream', prefix='fa', icon_color='white'),
            tooltip=f"🍦 #{route_num}: {row['Store Name']}"
        ).add_to(m)
        
        route_num += 1
    
    # Compact legend
    legend_html = '''
    <div style="position: fixed; 
                bottom: 20px; right: 20px; width: 180px; 
                background: rgba(255,255,255,0.97); 
                border: 3px solid #667eea; z-index:9999; 
                font-size:12px; padding: 10px; border-radius: 10px;
                box-shadow: 0 4px 8px rgba(0,0,0,0.4);">
        <p style="margin:0 0 8px 0; font-weight: bold; text-align: center; 
                  font-size: 14px; color: #667eea; border-bottom: 2px solid #ddd; padding-bottom: 6px;">
            🍦 Ice Cream Trail
        </p>
        <p style="margin: 6px 0; font-size: 11px;">
            <i class="fa fa-home" style="color:red"></i> Home
        </p>
        <p style="margin: 6px 0; font-size: 11px;">
            <i class="fa fa-ice-cream" style="color:blue"></i> Ice Cream Stores
        </p>
        <p style="margin: 6px 0 4px 0; font-weight: bold; font-size: 11px; 
                  border-top: 1px solid #ddd; padding-top: 6px;">
            Distance Colors:
        </p>
        <p style="margin: 4px 0; font-size: 11px; display: flex; align-items: center;">
            <span style="display: inline-block; width: 30px; height: 5px; 
                         background: #00cc00; margin-right: 8px; border-radius: 2px;"></span>
            <span>0-5 miles</span>
        </p>
        <p style="margin: 4px 0; font-size: 11px; display: flex; align-items: center;">
            <span style="display: inline-block; width: 30px; height: 5px; 
                         background: #ff9900; margin-right: 8px; border-radius: 2px;"></span>
            <span>5-15 miles</span>
        </p>
        <p style="margin: 4px 0; font-size: 11px; display: flex; align-items: center;">
            <span style="display: inline-block; width: 30px; height: 5px; 
                         background: #ff6600; margin-right: 8px; border-radius: 2px;"></span>
            <span>15-30 miles</span>
        </p>
        <p style="margin: 4px 0; font-size: 11px; display: flex; align-items: center;">
            <span style="display: inline-block; width: 30px; height: 5px; 
                         background: #cc0000; margin-right: 8px; border-radius: 2px;"></span>
            <span>30+ miles</span>
        </p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # Title
    title_html = '''
    <div style="position: fixed; 
                top: 10px; left: 50%; transform: translateX(-50%); 
                background: rgba(255,255,255,0.97); 
                padding: 10px 25px; border-radius: 25px;
                box-shadow: 0 3px 10px rgba(0,0,0,0.3);
                z-index: 9999;">
        <h3 style="margin:0; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                   -webkit-background-clip: text; -webkit-text-fill-color: transparent;
                   background-clip: text; font-family: Arial, sans-serif;
                   text-align: center; font-size: 20px; font-weight: bold;">
            Ice Cream Trail
        </h3>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    print(f"\n✅ Map created with {successful_routes}/{len(stores_df)} actual driving routes!")
    
    return m

# Create the map
print("📍 Creating your Ice Cream Trail map with REAL driving routes...\n")
ice_cream_map = create_ice_cream_trail_map(home_coords, results_df)
print("\n🎉 Map ready for LinkedIn!")

ice_cream_map

In [0]:
# Save the map as index.html for GitHub Pages
import os

# Save to workspace so you can download it
output_path = '/Workspace/Users/ahmadehtesham10@gmail.com/index.html'

# Save the map
ice_cream_map.save(output_path)

print("✅ Map saved successfully!")
print(f"📁 File location: {output_path}")
print("\n🎉 READY FOR GITHUB!")
print("=" * 60)
print("\nYour map is saved as 'index.html' - perfect for GitHub Pages!")
print("\n📥 HOW TO DOWNLOAD:")
print("  1. Go to the Databricks file explorer (left sidebar)")
print("  2. Navigate to /Users/ahmadehtesham10@gmail.com/")
print("  3. Find 'index.html'")
print("  4. Click ⋮ menu → Download")
print("\n🚀 UPLOAD TO GITHUB:")
print("  1. Go to your GitHub repo: github.com/ahmadratth/ice-cream-trail-mapper")
print("  2. Click 'Add file' → 'Upload files'")
print("  3. Upload index.html")
print("  4. Commit changes")
print("\n🌐 ENABLE GITHUB PAGES:")
print("  1. Go to Settings → Pages")
print("  2. Source: Deploy from branch → main → /(root)")
print("  3. Save")
print("  4. Your live map: https://ahmadratth.github.io/ice-cream-trail-mapper/")
print("\n" + "=" * 60)

In [0]:
import pandas as pd

# Create summary table with round-trip distances
summary_df = results_df.copy()
summary_df['Round-Trip (miles)'] = summary_df['Distance (miles)'] * 2
summary_df['Round-Trip (km)'] = summary_df['Distance (km)'] * 2

# Select and rename columns for display
display_df = summary_df[['Store Name', 'Distance (miles)', 'Round-Trip (miles)', 'Distance (km)', 'Round-Trip (km)']].copy()
display_df = display_df.rename(columns={
    'Distance (miles)': 'One-Way (mi)',
    'Round-Trip (miles)': 'Round-Trip (mi)',
    'Distance (km)': 'One-Way (km)',
    'Round-Trip (km)': 'Round-Trip (km)'
})

# Add rank
display_df.insert(0, 'Rank', range(1, len(display_df) + 1))

print("🚗 ICE CREAM TRAIL - ROUND-TRIP DISTANCE SUMMARY 🍦")
print("=" * 80)
print(f"\nTotal Stores: {len(display_df)}")
print(f"Total One-Way Distance (visiting all stores): {summary_df['Distance (miles)'].sum():.1f} miles")
print(f"Total Round-Trip Distance (visiting all stores): {summary_df['Round-Trip (miles)'].sum():.1f} miles")
print("\n" + "=" * 80)
print("\nDetailed Breakdown:\n")

# Display the table
display(display_df)

# Calculate additional stats
print("\n" + "=" * 80)
print("📊 STATISTICS:")
print(f"  Average One-Way Distance: {summary_df['Distance (miles)'].mean():.1f} miles")
print(f"  Average Round-Trip Distance: {summary_df['Round-Trip (miles)'].mean():.1f} miles")
print(f"  Closest Store: {display_df.iloc[0]['Store Name']} ({display_df.iloc[0]['Round-Trip (mi)']} miles round-trip)")
print(f"  Farthest Store: {display_df.iloc[-1]['Store Name']} ({display_df.iloc[-1]['Round-Trip (mi)']} miles round-trip)")
print("\n" + "=" * 80)

In [0]:
# Find the closest store
closest_store = results_df.iloc[0]

print("🍦 CLOSEST ICE CREAM STORE 🍦")
print("=" * 50)
print(f"Store Name: {closest_store['Store Name']}")
print(f"Address: {closest_store['Address']}")
print(f"Distance: {closest_store['Distance (miles)']} miles ({closest_store['Distance (km)']} km)")
print("=" * 50)

In [0]:
# Save to Delta table for future analysis
spark_df = spark.createDataFrame(results_df)

# Write to Delta table
spark_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ice_cream_stores_distances")

print("✅ Results saved to Delta table: ice_cream_stores_distances")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Get closest and farthest stores
closest_store = results_df.iloc[0]
farthest_store = results_df.iloc[-1]

# 1. Bar chart - Distance in miles
axes[0, 0].barh(results_df['Store Name'], results_df['Distance (miles)'], color='skyblue')
axes[0, 0].set_xlabel('Distance (miles)')
axes[0, 0].set_title('Distance to Each Ice Cream Store (miles)')
axes[0, 0].invert_yaxis()

# 2. Bar chart - Distance in km
axes[0, 1].barh(results_df['Store Name'], results_df['Distance (km)'], color='lightcoral')
axes[0, 1].set_xlabel('Distance (km)')
axes[0, 1].set_title('Distance to Each Ice Cream Store (km)')
axes[0, 1].invert_yaxis()

# 3. Pie chart - Distance distribution
axes[1, 0].pie(results_df['Distance (miles)'], labels=results_df['Store Name'], 
               autopct='%1.1f%%', startangle=90)
axes[1, 0].set_title('Distance Distribution')

# 4. Summary statistics
axes[1, 1].axis('off')
summary_text = f"""
SUMMARY STATISTICS
{'='*40}

Total Stores Analyzed: {len(results_df)}

Closest Store:
  • {closest_store['Store Name']}
  • {closest_store['Distance (miles)']} miles
  • {closest_store['Distance (km)']} km

Average Distance: {results_df['Distance (miles)'].mean():.2f} miles
Median Distance: {results_df['Distance (miles)'].median():.2f} miles

Farthest Store:
  • {farthest_store['Store Name']}
  • {farthest_store['Distance (miles)']} miles
  • {farthest_store['Distance (km)']} km

Stores within 10 miles: {len(results_df[results_df['Distance (miles)'] <= 10])}
Stores within 20 miles: {len(results_df[results_df['Distance (miles)'] <= 20])}
"""
axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
                verticalalignment='center')

plt.tight_layout()
plt.show()

##Alternative approach

In [0]:
# %pip
# install geopy openrouteservice

# import openrouteservice as ors
# from geopy.geocoders import Nominatim

# # Get free API key from: https://openrouteservice.org/dev/#/signup
# ORS_API_KEY = "YOUR_ORS_API_KEY"
# client = ors.Client(key=ORS_API_KEY)

# def calculate_route_ors(home, stores):
#     geolocator = Nominatim(user_agent="ice_cream_finder")
    
#     # Geocode home
#     home_location = geolocator.geocode(home)
#     home_coords = [home_location.longitude, home_location.latitude]
    
#     results = []
#     for store in stores:
#         store_location = geolocator.geocode(store['address'])
#         store_coords = [store_location.longitude, store_location.latitude]
        
#         # Get route
#         route = client.directions(
#             coordinates=[home_coords, store_coords],
#             profile='driving-car',
#             format='geojson'
#         )
        
#         distance_km = route['features'][0]['properties']['segments'][0]['distance'] / 1000
#         duration_min = route['features'][0]['properties']['segments'][0]['duration'] / 60
        
#         results.append({
#             'Store Name': store['name'],
#             'Address': store['address'],
#             'Distance (km)': round(distance_km, 2),
#             'Distance (miles)': round(distance_km * 0.621371, 2),
#             'Duration (minutes)': round(duration_min, 2)
#         })
    
#     return pd.DataFrame(results).sort_values('Duration (minutes)')